In [ ]:
# Fabric-native parameters and config
# Edit these values directly in Fabric, or pass them from a Fabric Data Pipeline.
ENV = "dev"
SCHEMA = "dbo"
MARKET = "AU"
MOCK_IPSTACK = True
LOOKBACK_DAYS = 90
LISTINGS_FILE = "listings.csv"
VIEWS_FILE = "property_views_5k.csv"

try:
    from notebookutils import mssparkutils
except ImportError:
    mssparkutils = None

from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

class Config:
    def __init__(self):
        self.env = ENV
        self.schema = SCHEMA
        self.market = MARKET
        self.mock_ipstack = MOCK_IPSTACK
        self.lookback_days = LOOKBACK_DAYS
        self.listings_file = LISTINGS_FILE
        self.views_file = VIEWS_FILE
        self.files_base = "Files"  # Fabric attached Lakehouse path
        self.listings_path = f"{self.files_base}/raw/sample/{self.listings_file}"
        self.views_path = f"{self.files_base}/raw/sample/{self.views_file}"
        self.fixture_base = f"{self.files_base}/fixtures/ipstack"

        self.bronze_listings = "bronze_listings"
        self.bronze_property_views = "bronze_property_views"
        self.bronze_ipstack_raw = "bronze_ipstack_raw"
        self.bronze_ipstack_errors = "bronze_ipstack_errors"
        self.silver_listings = "silver_listings"
        self.silver_ip_dim = "silver_ip_dim"
        self.silver_visits_enriched = "silver_visits_enriched"
        self.gold_suburb_interest = "gold_suburb_interest"
        self.gold_property_type_by_suburb = "gold_property_type_by_suburb"
        self.gold_price_engagement = "gold_price_engagement"
        self.gold_conversion_gaps = "gold_conversion_gaps"
        self.gold_property_trends = "gold_property_trends"
        self.gold_region_type_preference = "gold_region_type_preference"
        self.gold_repeat_interest = "gold_repeat_interest"
        self.gold_interstate_flow = "gold_interstate_flow"

    def table_fqn(self, table: str) -> str:
        return f"{self.schema}.{table}"

conf = Config()
print(f"env={conf.env} market={conf.market} mock={conf.mock_ipstack} views={conf.views_file}")
print(f"listings_path={conf.listings_path}")
print(f"views_path={conf.views_path}")



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

class GoldBuilder:
    """
    Gold star schema + business marts.

    Star schema grain:
      fact_property_view = one row per property view event.

    Marts are rebuilt from the fact table and dimensions so future fields can be
    added to dimensions without rewriting every dashboard table.
    """
    def __init__(self):
        self.c = conf
        self.v = spark.table(self.c.table_fqn(self.c.silver_visits_enriched))
        self.listings = spark.table(self.c.table_fqn(self.c.silver_listings))

    def table(self, name):
        return self.c.table_fqn(name)

    def key(self, *cols):
        return F.abs(F.crc32(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]))).cast("bigint")

    # ---------------------------- maintenance ----------------------------
    def gold_tables(self):
        return [
            "fact_property_view",
            "dim_date",
            "dim_time",
            "dim_property",
            "dim_suburb",
            "dim_visitor_geo",
            "dim_price_band",
            "dim_device",
            self.c.gold_suburb_interest,
            self.c.gold_property_type_by_suburb,
            self.c.gold_price_engagement,
            self.c.gold_conversion_gaps,
            self.c.gold_property_trends,
            self.c.gold_region_type_preference,
            self.c.gold_repeat_interest,
            self.c.gold_interstate_flow,
        ]

    def drop_gold_tables(self):
        for table in self.gold_tables():
            spark.sql(f"DROP TABLE IF EXISTS {self.table(table)}")
            print(f"dropped if exists: {self.table(table)}")

    # ---------------------------- dimensions ----------------------------
    def build_dim_date(self):
        df = (self.v.select(F.to_date("event_ts").alias("date"))
              .dropDuplicates()
              .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
              .withColumn("year", F.year("date"))
              .withColumn("month", F.month("date"))
              .withColumn("month_name", F.date_format("date", "MMMM"))
              .withColumn("week_of_year", F.weekofyear("date"))
              .withColumn("day_of_week", F.date_format("date", "EEEE"))
              .select("date_key", "date", "year", "month", "month_name", "week_of_year", "day_of_week"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("dim_date"))

    def build_dim_time(self):
        df = (self.v.select(F.hour("event_ts").alias("hour"))
              .dropDuplicates()
              .withColumn("time_key", F.col("hour"))
              .withColumn("time_period",
                  F.when(F.col("hour").between(5, 11), "morning")
                   .when(F.col("hour").between(12, 16), "afternoon")
                   .when(F.col("hour").between(17, 21), "evening")
                   .otherwise("night"))
              .select("time_key", "hour", "time_period"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("dim_time"))

    def build_dim_property(self):
        df = (self.listings
              .withColumn("property_key", self.key("property_id"))
              .select("property_key", "property_id", "property_type", "bedrooms", "bathrooms", "land_size_sqm", "is_luxury")
              .dropDuplicates(["property_key"]))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("dim_property"))

    def build_dim_suburb(self):
        df = (self.listings
              .withColumn("suburb_key", F.abs(F.crc32(F.concat_ws("|", F.col("suburb"), F.col("state")))).cast("bigint"))
              .select("suburb_key", "suburb", "postcode", F.col("state").alias("listing_state"))
              .dropDuplicates(["suburb_key"]))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("dim_suburb"))

    def build_dim_visitor_geo(self):
        df = (self.v
              .withColumn("visitor_geo_key", self.key("visitor_country", "visitor_region", "visitor_city", "visitor_state", "timezone_id"))
              .select("visitor_geo_key", "visitor_country", "visitor_region", "visitor_city", "visitor_state", "timezone_id")
              .dropDuplicates(["visitor_geo_key"]))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("dim_visitor_geo"))

    def build_dim_price_band(self):
        df = (self.v.select("price_range")
              .dropna()
              .dropDuplicates()
              .withColumn("price_band_key", self.key("price_range"))
              .withColumn("min_price",
                  F.when(F.col("price_range") == "under-500k", 0)
                   .when(F.col("price_range") == "500k-700k", 500000)
                   .when(F.col("price_range") == "700k-1M", 700000)
                   .when(F.col("price_range") == "1M-2M", 1000000)
                   .when(F.col("price_range") == "2M+", 2000000))
              .withColumn("max_price",
                  F.when(F.col("price_range") == "under-500k", 499999)
                   .when(F.col("price_range") == "500k-700k", 699999)
                   .when(F.col("price_range") == "700k-1M", 999999)
                   .when(F.col("price_range") == "1M-2M", 1999999)
                   .otherwise(F.lit(None).cast("int")))
              .select("price_band_key", "price_range", "min_price", "max_price"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("dim_price_band"))

    def build_dim_device(self):
        df = (self.v.select("device_type")
              .dropna()
              .dropDuplicates()
              .withColumn("device_key", self.key("device_type"))
              .select("device_key", "device_type"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("dim_device"))

    # ---------------------------- fact ----------------------------
    def build_fact_property_view(self):
        interstate = F.col("visitor_state") != F.col("listing_state")
        overseas = F.col("visitor_country") != F.lit("Australia")
        df = (self.v
              .withColumn("date_key", F.date_format(F.to_date("event_ts"), "yyyyMMdd").cast("int"))
              .withColumn("time_key", F.hour("event_ts"))
              .withColumn("property_key", self.key("property_id"))
              .withColumn("suburb_key", F.abs(F.crc32(F.concat_ws("|", F.col("suburb"), F.col("listing_state")))).cast("bigint"))
              .withColumn("visitor_geo_key", self.key("visitor_country", "visitor_region", "visitor_city", "visitor_state", "timezone_id"))
              .withColumn("price_band_key", self.key("price_range"))
              .withColumn("device_key", self.key("device_type"))
              .withColumn("is_interstate_view", F.when(interstate, True).otherwise(False))
              .withColumn("is_overseas_view", F.when(overseas, True).otherwise(False))
              .select("event_id", "session_id", "date_key", "time_key", "property_key", "suburb_key",
                      "visitor_geo_key", "price_band_key", "device_key", "property_id", "event_ts",
                      "view_duration_sec", "enquiry_flag", "favorite_flag", "is_interstate_view",
                      "is_overseas_view", "price"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table("fact_property_view"))

    def build_star_schema(self):
        self.build_dim_date()
        self.build_dim_time()
        self.build_dim_property()
        self.build_dim_suburb()
        self.build_dim_visitor_geo()
        self.build_dim_price_band()
        self.build_dim_device()
        self.build_fact_property_view()

    # ---------------------------- marts from star schema ----------------------------
    def load_star(self):
        fact = spark.table(self.table("fact_property_view"))
        suburb = spark.table(self.table("dim_suburb"))
        prop = spark.table(self.table("dim_property"))
        geo = spark.table(self.table("dim_visitor_geo"))
        price = spark.table(self.table("dim_price_band"))
        date = spark.table(self.table("dim_date"))
        return fact, suburb, prop, geo, price, date

    def suburb_interest(self):
        fact, suburb, _, _, _, _ = self.load_star()
        df = (fact.join(suburb, "suburb_key", "left")
              .groupBy("suburb", "listing_state")
              .agg(F.count("*").alias("total_views"),
                   F.sum(F.when(F.col("is_interstate_view"), 1).otherwise(0)).alias("interstate_views"),
                   F.sum(F.when(F.col("is_overseas_view"), 1).otherwise(0)).alias("overseas_views"),
                   F.countDistinct("session_id").alias("unique_sessions")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_suburb_interest))

    def property_type_by_suburb(self):
        fact, suburb, prop, _, _, _ = self.load_star()
        base = (fact.join(suburb, "suburb_key", "left")
                .join(prop, "property_key", "left")
                .groupBy("suburb", "property_type")
                .agg(F.count("*").alias("cnt")))
        w = Window.partitionBy("suburb").orderBy(F.desc("cnt"))
        df = (base.withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn") == 1)
              .select("suburb", F.col("property_type").alias("top_property_type"), "cnt"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_property_type_by_suburb))

    def price_engagement(self):
        fact, _, _, _, price, _ = self.load_star()
        df = (fact.join(price, "price_band_key", "left")
              .groupBy("price_range")
              .agg(F.avg("view_duration_sec").alias("avg_view_time_sec"),
                   F.avg(F.when(F.col("enquiry_flag"), 1.0).otherwise(0.0)).alias("enquiry_rate"),
                   F.count("*").alias("views")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_price_engagement))

    def conversion_gaps(self):
        fact, suburb, _, _, _, _ = self.load_star()
        df = (fact.join(suburb, "suburb_key", "left")
              .groupBy("suburb")
              .agg(F.count("*").alias("views"),
                   F.sum(F.when(F.col("enquiry_flag"), 1).otherwise(0)).alias("enquiries"))
              .withColumn("conversion_rate", F.col("enquiries") / F.col("views"))
              .orderBy(F.desc("views")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_conversion_gaps))

    def property_trends(self):
        fact, suburb, _, _, _, date = self.load_star()
        monthly = (fact.join(suburb, "suburb_key", "left")
                   .join(date, "date_key", "left")
                   .groupBy("suburb", F.date_trunc("month", F.col("date")).alias("month"))
                   .agg(F.count("*").alias("monthly_views")))
        w = Window.partitionBy("suburb").orderBy("month")
        df = monthly.withColumn("previous_month_views", F.lag("monthly_views").over(w))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_property_trends))

    def region_type_preference(self):
        fact, _, prop, geo, _, _ = self.load_star()
        base = (fact.join(prop, "property_key", "left")
                .join(geo, "visitor_geo_key", "left")
                .filter(F.col("visitor_state").isNotNull())
                .groupBy("visitor_state", "property_type")
                .agg(F.count("*").alias("views")))
        w = Window.partitionBy("visitor_state").orderBy(F.desc("views"))
        df = (base.withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn") == 1)
              .select(F.col("visitor_state").alias("visitor_region"),
                      F.col("property_type").alias("preferred_type"), "views"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_region_type_preference))

    def repeat_interest(self):
        fact, suburb, _, _, _, _ = self.load_star()
        df = (fact.join(suburb, "suburb_key", "left")
              .groupBy("session_id", "suburb")
              .agg(F.count("*").alias("repeated_views"),
                   F.max("enquiry_flag").alias("ever_enquired"))
              .filter(F.col("repeated_views") >= 5)
              .orderBy(F.desc("repeated_views")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_repeat_interest))

    def interstate_flow(self):
        fact, suburb, _, geo, _, _ = self.load_star()
        df = (fact.join(suburb, "suburb_key", "left")
              .join(geo, "visitor_geo_key", "left")
              .filter(F.col("visitor_state") != F.col("listing_state"))
              .groupBy(F.col("visitor_state"), F.col("listing_state").alias("viewed_state"))
              .agg(F.count("*").alias("views"), F.countDistinct("session_id").alias("sessions")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.table(self.c.gold_interstate_flow))

    def build_marts(self):
        self.suburb_interest()
        self.property_type_by_suburb()
        self.price_engagement()
        self.conversion_gaps()
        self.property_trends()
        self.region_type_preference()
        self.repeat_interest()
        self.interstate_flow()

    def run(self, reset_gold=True):
        if reset_gold:
            self.drop_gold_tables()
        self.build_star_schema()
        self.build_marts()
        print("✅ Gold star schema and marts built")

    def validate(self):
        tables = [
            "dim_date", "dim_time", "dim_property", "dim_suburb", "dim_visitor_geo",
            "dim_price_band", "dim_device", "fact_property_view",
            self.c.gold_suburb_interest, self.c.gold_conversion_gaps, self.c.gold_property_trends
        ]
        for name in tables:
            print(f"🔎 {name}: {spark.table(self.table(name)).count()} rows")

builder = GoldBuilder()
builder.run()
builder.validate()


